# insurance-monitoring

**Model monitoring for insurance pricing: PSI, A/E ratios, Gini drift test, and Murphy decomposition — in one report.**

Your aggregate A/E ratio looks fine. Your model has been mispricing under-25s for eight months. This notebook shows why PSI per feature catches what aggregate A/E misses, and how the Murphy decomposition tells you whether to RECALIBRATE (cheap) or REFIT (expensive).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-monitoring/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-monitoring polars numpy

## Synthetic monitoring scenario

We simulate a UK motor pricing model trained on 2021–2022 data, now monitoring a 2024 portfolio with three induced failure modes:

1. **Covariate shift**: young drivers (under-30) are 2x oversampled in the monitoring period
2. **Calibration drift**: actual claims running 8% above model predictions
3. **A/E stable at aggregate level** — the two effects partially cancel, masking the problem

The monitoring report should detect all three.

In [ ]:
import numpy as np
import polars as pl

rng = np.random.default_rng(42)

# Reference period (training window): 50,000 policies, well-calibrated
n_ref = 50_000
pred_ref = rng.uniform(0.05, 0.20, n_ref)
act_ref  = rng.poisson(pred_ref).astype(float)

feat_ref = pl.DataFrame({
    "driver_age":  rng.integers(18, 80, n_ref).tolist(),
    "vehicle_age": rng.integers(0, 15, n_ref).tolist(),
    "ncd_years":   rng.integers(0, 9, n_ref).tolist(),
})

# Current monitoring period: 15,000 policies, with induced shifts
n_cur = 15_000
# Young driver oversampling: 60% under-30 (vs ~17% in reference)
young_ages   = rng.integers(18, 30, int(n_cur * 0.60))
mature_ages  = rng.integers(30, 80, n_cur - int(n_cur * 0.60))
driver_ages_cur = np.concatenate([young_ages, mature_ages])
rng.shuffle(driver_ages_cur)

pred_cur = rng.uniform(0.05, 0.20, n_cur)
act_cur  = rng.poisson(pred_cur * 1.08).astype(float)  # 8% calibration drift

feat_cur = pl.DataFrame({
    "driver_age":  driver_ages_cur.tolist(),
    "vehicle_age": rng.integers(0, 15, n_cur).tolist(),
    "ncd_years":   rng.integers(0, 9, n_cur).tolist(),
})

print(f"Reference: {n_ref:,} policies, A/E = {act_ref.sum() / pred_ref.sum():.3f}")
print(f"Current:   {n_cur:,} policies, A/E = {act_cur.sum() / pred_cur.sum():.3f}")
print(f"Under-30 share: reference {(np.array(feat_ref['driver_age'].to_list()) < 30).mean():.0%}, "
      f"current {(driver_ages_cur < 30).mean():.0%}")

## Run the monitoring report

`MonitoringReport` runs PSI/CSI for feature distributions, A/E ratio with Poisson CIs, the Gini drift z-test, and the Murphy decomposition — all in one call. The `recommendation` property gives you the action: NO_ACTION, MONITOR_CLOSELY, RECALIBRATE, INVESTIGATE, or REFIT.

In [ ]:
from insurance_monitoring import MonitoringReport

report = MonitoringReport(
    reference_actual=act_ref,
    reference_predicted=pred_ref,
    current_actual=act_cur,
    current_predicted=pred_cur,
    feature_df_reference=feat_ref,
    feature_df_current=feat_cur,
    features=["driver_age", "vehicle_age", "ncd_years"],
    murphy_distribution="poisson",
)

print(f"Recommendation: {report.recommendation}")
print()
print(report.to_polars())

## What you should see

| metric | value | interpretation |
|--------|-------|----------------|
| ae_ratio | ~1.08 | AMBER — model underpredicts by 8% |
| csi_driver_age | >0.25 | RED — young driver oversampling is large |
| gini_current | varies | may flag if ranking degraded |
| murphy_miscalibration | >0 | RECALIBRATE — intercept shift, not broken ranking |

The key insight: aggregate A/E alone would give you "investigate" but not tell you *why*. CSI on `driver_age` immediately points to the covariate shift. Murphy decomposition tells you whether the fix is cheap (scale the intercept) or expensive (rebuild the model).

## RECALIBRATE vs REFIT: the Murphy decision

If GMCB (global miscalibration) > LMCB (local miscalibration), a simple multiplier correction fixes it: RECALIBRATE. If LMCB >= GMCB, the model's ranking is broken and a full refit is needed. This is the difference between hours of work and weeks.

In [ ]:
from insurance_monitoring.calibration import murphy_decomposition

result = murphy_decomposition(act_cur, pred_cur, distribution="poisson")
print(f"Discrimination (DSC): {result.discrimination:.4f}")
print(f"Miscalibration (MCB): {result.miscalibration:.4f}")
print(f"Global MCB (GMCB):    {result.global_mcb:.4f}  <- fixable by multiplying all preds by A/E")
print(f"Local MCB (LMCB):     {result.local_mcb:.4f}  <- requires model refit if dominant")
print(f"Verdict:              {result.verdict}")

## ModelMonitor — the v1.0.0 integrated decision framework

`ModelMonitor` is the recommended API for teams that want a single, structured answer rather than a collection of individual metrics. It runs three tests from Brauer, Menzel & Wüthrich (2025), arXiv:2510.04556:

- **Gini ranking drift test** (Algorithm 3): has the model's rank-ordering degraded?
- **GMCB global calibration test** (Algorithm 4a): is there a flat level shift fixable by a multiplier?
- **LMCB local calibration test** (Algorithm 4b): are specific cohorts badly miscalibrated?

The three-way decision:

| Condition | Decision | What it means |
|-----------|----------|---------------|
| Nothing fires | REDEPLOY | Model is fine. Re-run next quarter. |
| Only GMCB | RECALIBRATE | Multiply predictions by A/E ratio. Hours of work. |
| Gini OR LMCB | REFIT | Rebuild the model. Weeks of work. |

Alpha=0.32 (one-sigma rule) is the paper's recommendation for ongoing production monitoring. A false alarm costs a model review; a missed detection costs mis-pricing.

In [ ]:
from insurance_monitoring import ModelMonitor

# Fit on the reference period data we already created above
monitor = ModelMonitor(
    distribution='poisson',
    n_bootstrap=400,    # 400 is fast; use 1000+ for governance packs
    alpha_gini=0.32,
    alpha_global=0.32,
    alpha_local=0.32,
    random_state=42,
)
monitor.fit(act_ref, pred_ref)

# Test on the current monitoring period
result = monitor.test(act_cur, pred_cur)

print(f"Decision: {result.decision}")
print(f"Reason:   {result.decision_reason}")
print()
print(f"Gini ranking test:   z={result.gini_z:.2f}  p={result.gini_p:.3f}  significant={result.gini_significant}")
print(f"GMCB global calib:   score={result.gmcb_score:.5f}  p={result.gmcb_p:.3f}  significant={result.gmcb_significant}")
print(f"LMCB local calib:    score={result.lmcb_score:.5f}  p={result.lmcb_p:.3f}  significant={result.lmcb_significant}")
print(f"Balance factor:      {result.balance_factor:.3f}  (multiply predictions by this for RECALIBRATE)")
print()
print("Governance summary:")
print(result.summary())

### Simulating the RECALIBRATE scenario

The monitoring period above shows 8% calibration drift on a model with flat predictions (no feature signal). Let's simulate a scenario where the frequency trend has moved but ranking is intact — the canonical RECALIBRATE case.

In [ ]:
# Global frequency inflation: true claims running 12% higher, rank structure intact
# This happens in practice after hard market conditions or a cold winter.
act_inflated = rng.poisson(pred_ref * 1.12).astype(float)  # 12% trend, same ranking

monitor_fresh = ModelMonitor(distribution='poisson', n_bootstrap=400, random_state=0)
monitor_fresh.fit(act_ref, pred_ref)
result_recalib = monitor_fresh.test(act_inflated, pred_ref)

print(f"Decision: {result_recalib.decision}")
print(f"Balance factor: {result_recalib.balance_factor:.3f}")
if result_recalib.decision == 'RECALIBRATE':
    print(f"  -> Multiply all predictions by {result_recalib.balance_factor:.3f} before the next renewal cycle.")
    print(f"     No model rebuild needed.")

# to_dict() gives a JSON-serialisable record for MLflow or Delta table logging
d = result_recalib.to_dict()
print(f"\nFor MLflow logging: result.to_dict() has {len(d)} fields")
print(f"  decision={d['decision']}, gini_p={d['gini_p']:.3f}, gmcb_p={d['gmcb_p']:.3f}")

## Next steps

- **`ae_ratio_ci()`** — Garwood exact Poisson confidence intervals on A/E by segment
- **`gini_drift_test()`** — statistical test for Gini degradation (arXiv 2510.04556 Theorem 1)
- **`MonitoringThresholds`** — tighten PSI thresholds for large motor books with monthly monitoring
- **`CalibrationChecker`** — full sign-off suite: balance test, auto-calibration, Murphy report

**GitHub:** https://github.com/burning-cost/insurance-monitoring  
**PyPI:** https://pypi.org/project/insurance-monitoring/